# 预训练的数据流水线

模型是一面镜子，它反映的是你喂给它的东西，如果你喂垃圾给它，那它也会吐出垃圾给你。

## 问题描述

你需要喂给分词器的数据需要是，干净的、去重的、高质量的、分词成定长序列、且能够以随机批次进行供给的。

很多人认为训练一个大模型是关于模型架构的问题，其实不然。基本上所有的大模型底层都是相似的：堆叠的带注意力和前馈网络的Tranformer块。训练时使用的数据决定了模型的输出质量。

Chinchilla 的研究结果表明，模型参数量和训练使用的词元量之间存在最优的比例。过往的很多模型都处于欠训练的状态。

## 基本概念

### 数据来源

基本上所有的大语言模型训练的语料都是混合的，包括网络数据、代码、数据、学术论文等等。。。

各种训练语料之间的比例很重要。过多的网络数据会使模型变成一只学舌的鹦鹉，过少的代码数据它就不能够执行编码任务等等。。。

### 数据清洗

一个典型的文本噪音会包括：
* HTML 标记和JavaScript 脚本
* 样板化的业头、业脚、导航菜单等
* 机器生成的垃圾
* 用户个人信息（PII： Personally indentifiable information）
* 低质量的文本
* 编码成文本的非文本内容

因此一个典型的清洗流程为：
```mermaid
flowchart LR
A[原始文本]-->B[清理HTML]-->C[检测语言类别]-->D[按质量过滤]-->E[去重]-->F[去除个人信息]-->G[干净文本]
```

#### 清理HTML

移除所有的HTML标记，仅保留可视的文本内容。

#### 语言检测

使用fastText 检测文本的语言类型和置信度。一篇检测为英文但是置信度只有0.8的文本不是干净的。

#### 质量过滤

在高质量的语料比如维基百科上训练一个打分模型，然后对文本进行过滤。

#### 去重

最重要的一步。在重复的语料上训练浪费算力，而且容易让模型学会死记硬背。

#### 去除个人信息

名称、邮箱、电话号码等信息。通常通过正则检测，或者支持PII，NER任务的大模型

### 去重 -- MinHash

精确去重很简单，按哈希分桶，然后去重即可。但是近似重复才是真正棘手的问题，比如报道了同一则新闻但是周围的广告不同，就是近似重复的一种。大部分内容都是重复的，但是精确匹配去重不生效。

采用MinHash + Locality-Sensitive Hashing （LSH）处理。
1. 切片。将每篇文档转换成一组n-gram切片。
2. MinHash。 对于每篇文档的切片集合，计算k个哈希值，每个哈希值对应不同的哈希函数。这构成一个定长的“签名”，近似任意两篇文档之间的Jaccard相似度。
3. LSH。 根据生产的“签名”进行分桶。
4. 验证。 对桶内的两两候选，计算Jaccard相似度，然后做去重。

### 序列打包

模型期望输入的是定长序列，但是文本的长度是不定的，有些只有50个词元，有些则有几万个。

常规的做法是，将每篇文本用Pad进行填充到指定长度。但是这会浪费巨大的算力在计算Pad上。

**聪明的做法则是，将多篇文本打包成一个单独序列。然后使用特殊标记对词元进行分隔。**

此时，注意力掩码必须被合理设置。文本B的词元不应该注意到文本A重的内容。

### Chinchilla 缩放定律

在给定算力的情况下，最优的模型大小`N` 和训练语料大小`D`应该满足：
```
N_opt ~ C^0.5
D_opt ~ C^0.5
```

# 开始构建